In [10]:
import requests
from dotenv import load_dotenv
import boto3
from pathlib import Path
import pandas as pd
import json

load_dotenv()  # reads .env, sets env variables

s3 = boto3.client("s3")
bucket_name = "weather-data-eng" #bucket_name

tokyo_df = pd.read_json(s3.get_object(
            Bucket=bucket_name,
            Key="bronze/weather/tokyo/tokyo_backfill_2023-07_to_2026-07.json")['Body'])

tokyo_df.head(20)



,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,hourly_units,hourly
time,35.7,139.6875,7233.748555,0,GMT,GMT,40,iso8601,"[2023-07-01T00:00, 2023-07-01T01:00, 2023-07-0..."
temperature_2m,35.7,139.6875,7233.748555,0,GMT,GMT,40,°C,"[25.6, 25.1, 24.2, 24.5, 25.5, 26.7, 27.8, 28...."
rain,35.7,139.6875,7233.748555,0,GMT,GMT,40,mm,"[0.4, 1.5, 7.3, 7.3, 4.0, 0.0, 0.0, 0.0, 0.0, ..."
uv_index,35.7,139.6875,7233.748555,0,GMT,GMT,40,,"[1.65, 1.1, 0.7000000000000001, 1.3, 0.55, 2.0..."
sunshine_duration,35.7,139.6875,7233.748555,0,GMT,GMT,40,s,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 333.86, 0...."


In [11]:
import requests
from dotenv import load_dotenv
import boto3
from pathlib import Path
import pandas as pd
import json

load_dotenv()

s3 = boto3.client("s3")
bucket_name = "weather-data-eng"

# Load the raw bronze JSON as a dict, not via read_json
response = s3.get_object(
    Bucket=bucket_name,
    Key="bronze/weather/tokyo/tokyo_backfill_2023-07_to_2026-07.json"
)
tokyo_data = json.loads(response["Body"].read())

# --- Checks ---

# 1. Confirm top-level structure looks right
print(tokyo_data.keys())

# 2. Confirm location matches what you expect
print("Lat:", tokyo_data.get("latitude"), "Lon:", tokyo_data.get("longitude"))

# 3. Confirm units are present
print(tokyo_data.get("hourly_units"))

# 4. Load the hourly data into a DataFrame to inspect it properly
tokyo_hourly_df = pd.DataFrame(tokyo_data["hourly"])
print(tokyo_hourly_df.shape)
print(tokyo_hourly_df.head(20))

# 5. Confirm the date range covers what you expect (3 years)
print("Earliest:", tokyo_hourly_df["time"].min())
print("Latest:", tokyo_hourly_df["time"].max())

# 6. Check for any obviously missing/null data
print(tokyo_hourly_df.isnull().sum())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
Lat: 35.7 Lon: 139.6875
{'time': 'iso8601', 'temperature_2m': '°C', 'rain': 'mm', 'uv_index': '', 'sunshine_duration': 's'}
(27048, 5)
                time  temperature_2m  rain  uv_index  sunshine_duration
0   2023-07-01T00:00            25.6   0.4      1.65               0.00
1   2023-07-01T01:00            25.1   1.5      1.10               0.00
2   2023-07-01T02:00            24.2   7.3      0.70               0.00
3   2023-07-01T03:00            24.5   7.3      1.30               0.00
4   2023-07-01T04:00            25.5   4.0      0.55               0.00
5   2023-07-01T05:00            26.7   0.0      2.05               0.00
6   2023-07-01T06:00            27.8   0.0      1.15               0.00
7   2023-07-01T07:00            28.0   0.0      0.45             333.86
8   2023-07-01T08:00            27.3   0.0      0.30         